# Interactive Linked Charts Demo

Two Plotly visualizations of the same data, with **synchronized zoom** —
drag to zoom on either chart and watch the other follow.

Each chart is a [metaframe](https://js.mtfm.io/) running its own JavaScript,
connected together via `pipe_to()`. Data is generated in Python and pushed to both.

In [ ]:
%pip install ../../python

In [ ]:
from metaframe_widget import MetaframeWidget

## 1 — Wave Components (line chart)

Individual sine-wave components rendered as colored lines.

In [ ]:
chart1 = MetaframeWidget.from_code("""
const s = document.createElement('script');
s.src = 'https://cdn.plot.ly/plotly-2.35.2.min.js';
document.head.appendChild(s);
await new Promise(r => s.onload = r);

root.style.cssText = 'width:100%;height:100%;background:#0d1117';
let _syncing = false, _listening = false;

const palette = ['#58a6ff','#f97583','#d2a8ff','#56d364','#e3b341','#79c0ff'];
const layout = {
  paper_bgcolor:'#0d1117', plot_bgcolor:'#161b22',
  font:{color:'#c9d1d9',size:11},
  title:{text:'Wave Components',font:{size:15,color:'#f0f6fc'}},
  xaxis:{gridcolor:'#21262d',title:'Time',zerolinecolor:'#30363d'},
  yaxis:{gridcolor:'#21262d',title:'Amplitude',zerolinecolor:'#30363d'},
  legend:{bgcolor:'rgba(13,17,23,0.8)',font:{color:'#c9d1d9',size:10},
          orientation:'h',y:-0.18},
  margin:{t:40,r:10,b:60,l:50},
};

function render(data) {
  const traces = Object.entries(data.series).map(([name, y], i) => ({
    x:data.x, y, name, type:'scatter', mode:'lines',
    line:{color:palette[i%palette.length], width:2, shape:'spline'},
  }));
  Plotly.react(root, traces, layout, {responsive:true});
  if (!_listening) {
    root.on('plotly_relayout', onZoom);
    _listening = true;
  }
}

function onZoom(e) {
  if (_syncing) return;
  if (e['xaxis.range[0]'] != null)
    setOutput('zoom',{xmin:e['xaxis.range[0]'],xmax:e['xaxis.range[1]']});
  else if (e['xaxis.autorange'])
    setOutput('zoom',{reset:true});
}

export const onInputs = (inputs) => {
  if (inputs.data) render(inputs.data);
  if (inputs.zoom) {
    _syncing = true;
    const u = inputs.zoom.reset
      ? {'xaxis.autorange':true}
      : {'xaxis.range':[inputs.zoom.xmin,inputs.zoom.xmax]};
    Plotly.relayout(root,u).then(()=>setTimeout(()=>{_syncing=false},50));
  }
};
""", height="420px")
chart1

## 2 — Combined Signal (area chart)

The sum of all components shown as a filled area, with a smoothed trend line.

In [ ]:
chart2 = MetaframeWidget.from_code("""
const s = document.createElement('script');
s.src = 'https://cdn.plot.ly/plotly-2.35.2.min.js';
document.head.appendChild(s);
await new Promise(r => s.onload = r);

root.style.cssText = 'width:100%;height:100%;background:#0d1117';
let _syncing = false, _listening = false;

function smooth(arr, w) {
  return arr.map((_, i) => {
    let sum = 0, n = 0;
    for (let j = Math.max(0,i-w); j <= Math.min(arr.length-1,i+w); j++) {
      sum += arr[j]; n++;
    }
    return sum / n;
  });
}

const layout = {
  paper_bgcolor:'#0d1117', plot_bgcolor:'#161b22',
  font:{color:'#c9d1d9',size:11},
  title:{text:'Combined Signal',font:{size:15,color:'#f0f6fc'}},
  xaxis:{gridcolor:'#21262d',title:'Time',zerolinecolor:'#30363d'},
  yaxis:{gridcolor:'#21262d',title:'Amplitude',zerolinecolor:'#30363d'},
  legend:{bgcolor:'rgba(13,17,23,0.8)',font:{color:'#c9d1d9',size:10},
          orientation:'h',y:-0.18},
  margin:{t:40,r:10,b:60,l:50},
  shapes:[],
};

function render(data) {
  const x = data.x, y = data.combined;
  const trend = smooth(y, 15);

  const area = {
    x, y, type:'scatter', mode:'lines',
    fill:'tozeroy',
    fillcolor:'rgba(249,115,131,0.12)',
    line:{color:'#f97583',width:1.5},
    name:'Combined',
  };
  const trendLine = {
    x, y:trend, type:'scatter', mode:'lines',
    line:{color:'#e3b341',width:2.5,shape:'spline'},
    name:'Trend',
  };

  layout.shapes = [{
    type:'line', x0:x[0], x1:x[x.length-1], y0:0, y1:0,
    line:{color:'#484f58',width:1,dash:'dot'},
  }];

  Plotly.react(root, [area, trendLine], layout, {responsive:true});
  if (!_listening) {
    root.on('plotly_relayout', onZoom);
    _listening = true;
  }
}

function onZoom(e) {
  if (_syncing) return;
  if (e['xaxis.range[0]'] != null)
    setOutput('zoom',{xmin:e['xaxis.range[0]'],xmax:e['xaxis.range[1]']});
  else if (e['xaxis.autorange'])
    setOutput('zoom',{reset:true});
}

export const onInputs = (inputs) => {
  if (inputs.data) render(inputs.data);
  if (inputs.zoom) {
    _syncing = true;
    const u = inputs.zoom.reset
      ? {'xaxis.autorange':true}
      : {'xaxis.range':[inputs.zoom.xmin,inputs.zoom.xmax]};
    Plotly.relayout(root,u).then(()=>setTimeout(()=>{_syncing=false},50));
  }
};
""", height="420px")
chart2

## 3 — Connect zoom & send data

Link the charts so zooming one zooms the other, then generate data and push it to both.

In [ ]:
# Bidirectional zoom sync
chart1.pipe_to(chart2, output_key="zoom", input_key="zoom")
chart2.pipe_to(chart1, output_key="zoom", input_key="zoom")

In [ ]:
import numpy as np

x = np.linspace(0, 4 * np.pi, 600)

series = {
    "\u03b1  2 Hz":    np.sin(2 * x).tolist(),
    "\u03b2  3.5 Hz":  (0.7 * np.sin(3.5 * x + 0.8)).tolist(),
    "\u03b3  7 Hz":    (0.35 * np.sin(7 * x + 2.0)).tolist(),
    "\u03b4  0.5 Hz":  (1.4 * np.sin(0.5 * x + 0.3)).tolist(),
}

combined = np.zeros_like(x)
for v in series.values():
    combined += np.array(v)

data = {"x": x.tolist(), "series": series, "combined": combined.tolist()}

chart1.set_input("data", data)
chart2.set_input("data", data)

print(f"Sent {len(x)} data points with {len(series)} wave components")

**Try it:** drag to zoom on either chart — the other chart follows. Double-click to reset.